# PARSEC isochrone → CSV

Downloads a [PARSEC](http://stev.oapd.inaf.it/cgi-bin/cmd) isochrone (CMD 3.8 web service, INAF–Padova) for a given age and metallicity and writes a CSV that can be overlaid on the colour–magnitude diagram of an open cluster.

Steps:

1. POST request to the CMD 3.8 service with the chosen log(age) and metallicity.
2. Parse the returned `.dat` file (Gaia EDR3 photometric system).
3. Extract the `Gmag`, `G_BPmag`, `G_RPmag` columns.
4. Keep evolutionary stages label = 0–5: 0 = pre-main sequence, 1 = main sequence, 2 = subgiants, 3 = red giants, 4 = core He burning (horizontal branch / red clump), 5 = blueward part of the Cepheid loop. Dropped: 6 (redward part of the Cepheid loop), 7 = early AGB, 8 = TP-AGB, 9 = post-AGB. Label meanings: CMD 3.8 help and [FAQ](https://stev.oapd.inaf.it/cmd_3.8/faq.html).
5. Write a CSV with columns:
   - `BP_RP` – intrinsic isochrone colour (mag)
   - `M_G` – absolute magnitude of the isochrone (mag)

Typical values of log(age) (literature ages — see the Gaia notebook):

| log(age) | Age | Example cluster (literature age) |
|---|---|---|
| 8.05 | 112 Myr | Pleiades (112 Myr; Dahm 2015) |
| 8.18 | 151 Myr | NGC 2516 (≈150 Myr; Bouma et al. 2021) |
| 8.80 | 631 Myr | Hyades (625 Myr; Perryman et al. 1998) |
| 9.13 | 1.35 Gyr | NGC 752 (1.34 Gyr; Agüeros et al. 2018) |
| 9.30 | 2.0 Gyr | |
| 9.60 | 3.98 Gyr | M67 (3.95 Gyr; Reyes et al. 2024) |

**Usage:** set the parameters in the settings cell and run all cells (Runtime → Run all). The CSV is downloaded automatically at the end.

References: Bressan et al. (2012); Marigo et al. (2017); bolometric corrections YBC — Chen et al. (2019, A&A 632, A105). Further references for the form parameters are given in the code cell.

In [ ]:
LOG_AGE = 9.00  #@param {type:"number"}
FEH = 0.0       #@param {type:"number"}
OUTPUT = ""     #@param {type:"string"}
# LOG_AGE: log10(age / yr), e.g. 8.00 = 100 Myr, 9.60 = 3.98 Gyr
# FEH: metallicity [Fe/H] (0.0 = solar)
# Leave OUTPUT empty to use isochrone_log<X>.csv

In [ ]:
import csv
import re
import sys
import tempfile
from typing import List, Optional, Tuple

import numpy as np
import requests

PARSEC_URL = "https://stev.oapd.inaf.it/cgi-bin/cmd_3.8"
PARSEC_BASE = "https://stev.oapd.inaf.it"

# The server stev.oapd.inaf.it does not send its intermediate CA certificate
# (ZeroSSL RSA DV SSL CA 2) in the TLS handshake, so OpenSSL/requests cannot
# build a chain to the root CA even though the certificate is valid (browsers
# hide this by fetching the missing intermediate themselves). This public
# intermediate certificate is appended to the CA bundle only as a fallback,
# when the default verification fails.
_ZEROSSL_INTERMEDIATE = """-----BEGIN CERTIFICATE-----
MIIGITCCBAmgAwIBAgIRANCpCCVfjlDl8zltERRwKjcwDQYJKoZIhvcNAQEMBQAw
XzELMAkGA1UEBhMCR0IxGDAWBgNVBAoTD1NlY3RpZ28gTGltaXRlZDE2MDQGA1UE
AxMtU2VjdGlnbyBQdWJsaWMgU2VydmVyIEF1dGhlbnRpY2F0aW9uIFJvb3QgUjQ2
MB4XDTI1MDkyNDAwMDAwMFoXDTM1MDkyMzIzNTk1OVowRjELMAkGA1UEBhMCQVQx
FTATBgNVBAoTDFplcm9TU0wgR21iSDEgMB4GA1UEAxMXWmVyb1NTTCBSU0EgRFYg
U1NMIENBIDIwggGiMA0GCSqGSIb3DQEBAQUAA4IBjwAwggGKAoIBgQCnXX3Qm/G+
ujylvYIkhJ53ZZ7XM03vXY03sCnO9VWjwMsV0qCQMlvhuRiJwrm4M5jgGehxIiCR
1qT0AyL2rHTWJR07vjJzuNmp7uoKu3HKwixBk9QuXD5aliO8/EDbzdZDcG/Hm5pC
mOeusLtds/UE/Iq24nw5WpcgZE5Ly+F/yCYDuEa7hXLJAPM5SecJIblG1OQ3ukMl
HHNBbDxBpGWTyDudd0DTed0NTgCPs1t8RZPhW9Gt/8nDwFX0pQ4eHfPEB8c6eNl2
sRr2Afp1YErakR+53yEX2SXc2Kz0fbTlUc+To0ULGcJiWNyZwj//DTZ+M4xxsT2T
qjQ4Xfvm2EUTymrXDrh1Pm/wkBouu860c6eeQfNlKlccUyHOSeKCtPIWreMvH3Be
Ydeu3DwI8lefn/VUhSB2Bbz7hX3qz3oMmtSTmWhTnobyKlx1L2b/oloaqpy1cBc/
QiLRSOptYGPjZtX0pRrTVKQXeP2rPUk0y5q/40WRpugSlHCX6aceWnsCAwEAAaOC
AW8wggFrMB8GA1UdIwQYMBaAFFZzWGSV+ZIasBIqBGJ5oUAViCFJMB0GA1UdDgQW
BBRLvvp2hCNEBLnOvjFv6fUyBv8MVzAOBgNVHQ8BAf8EBAMCAYYwEgYDVR0TAQH/
BAgwBgEB/wIBADATBgNVHSUEDDAKBggrBgEFBQcDATATBgNVHSAEDDAKMAgGBmeB
DAECATBUBgNVHR8ETTBLMEmgR6BFhkNodHRwOi8vY3JsLnNlY3RpZ28uY29tL1Nl
Y3RpZ29QdWJsaWNTZXJ2ZXJBdXRoZW50aWNhdGlvblJvb3RSNDYuY3JsMIGEBggr
BgEFBQcBAQR4MHYwTwYIKwYBBQUHMAKGQ2h0dHA6Ly9jcnQuc2VjdGlnby5jb20v
U2VjdGlnb1B1YmxpY1NlcnZlckF1dGhlbnRpY2F0aW9uUm9vdFI0Ni5wN2MwIwYI
KwYBBQUHMAGGF2h0dHA6Ly9vY3NwLnNlY3RpZ28uY29tMA0GCSqGSIb3DQEBDAUA
A4ICAQCJ/3v2/vdexHsdVyXL9aCTQE01YXl23866TVM/LgRpRW+kneZXXZxP0hy4
GnvlqUcxTq97B6qPdQcQxQxpGne7CRn0nWauzqieMcJzYl3fDC2Q/ANyPhyrbwCI
zx9EsRrgfjvuJCaUMtlfYpKqBUYiPOCPAN0HdrLD5hU6oV1tvWVsUzTA43skC3uH
wQM5YPIk0NDJFw3NhQPOIOwbq09T+SYSEZvsJ3t4sA4H3gh03RETNaAwTcTNS/+u
1tAeQUZZmKQyLWYLyoxvbISp/MFr9xqhDqrpAurYVNeiLJ5+4/WZPml20yNZjcxV
KKqRYdEurl8rmI2toCnCDWiEcTDvoYGtz60eYIt7VJID4DrCjTxAWTWh2T5ag2pK
ryNJGt8BFGLtjeD774SxAFn5MGYBVvEK4LXmCjVX78pb6/0Dceo71dlQUK68yftb
+yTeyoqjwgk2L8vNSrkj6UTkvvqXSONFuVU7bvC0O/9bi5MXBv7QUivMNxDsTtaT
IZO7PsSCRmLraQM2EPBgbNL9lRSEYi4Hj+NicT/e87pbhv88k0oec9xGcnkcvZpN
sJBz78mkZfeFIIjh02e2y9ke/Fqw1FbdhHtU2myaFnX0sRyGLmI/vzXSwyvT+Kxl
Wx8FV64z2PBdwd0vXRaAMXomGC0M1vQa5fBDn4dimzewsSiphQ==
-----END CERTIFICATE-----
"""

_fallback_ca_bundle = None


def _extended_ca_bundle() -> str:
    """Return the path to a CA bundle extended with the missing intermediate
    certificate (created once, lazily)."""
    global _fallback_ca_bundle
    if _fallback_ca_bundle is None:
        import certifi
        with tempfile.NamedTemporaryFile(
                "w", suffix=".pem", delete=False) as f:
            with open(certifi.where()) as bundle:
                f.write(bundle.read())
            f.write(_ZEROSSL_INTERMEDIATE)
            _fallback_ca_bundle = f.name
    return _fallback_ca_bundle


_CGIFIELDS = [
    "dust_sourceC", "dust_sourceM", "extinction_coeff", "extinction_curve",
    "isoc_isagelog", "isoc_ismetlog", "kind_LPV", "output_gzip",
    "output_kind", "photsys_version", "track_colibri",
    "track_parsec", "track_postagb",
]


def fetch_isochrone(log_age: float, feh: float = 0.0) -> Optional[Tuple[np.ndarray, np.ndarray]]:
    # Z_sun = 0.0152: present-day solar metal content in PARSEC
    # (Bressan et al. 2012, MNRAS 427, 127; also stated on the CMD 3.8 form).
    # Z = Z_sun * 10^[Fe/H] is an approximation for a scaled-solar mixture; CMD
    # 3.8 recommends [M/H] = log(Z/X) - log(Z/X)_sun, (Z/X)_sun = 0.0207,
    # Y = 0.2485 + 1.78 Z (for Z = 0.0152 CMD reports [M/H] = +0.015).
    Z_sun = 0.0152
    Z = Z_sun * 10.0**feh

    # CMD 3.8 form parameters (https://stev.oapd.inaf.it/cgi-bin/cmd_3.8);
    # references as listed in the header of the returned .dat file:
    #   track_parsec  - PARSEC v1.2S (Bressan et al. 2012; Chen et al. 2014,
    #                   MNRAS 444, 2525; Chen et al. 2015, MNRAS 452, 1068;
    #                   Tang et al. 2014, MNRAS 445, 4287)
    #   track_colibri - COLIBRI S_37 + S_35 + PR16 for the TP-AGB (Pastorelli
    #                   et al. 2019, MNRAS 485, 5666; 2020, MNRAS 498, 3283;
    #                   Marigo et al. 2013, MNRAS 434, 488)
    #   n_inTPC       - thermal-pulse resolution (Marigo et al. 2017, ApJ 835,
    #                   77); default value of the CMD 3.8 form
    #   eta_reimers   - Reimers mass-loss efficiency on the RGB; default value
    #                   of the CMD 3.8 form
    #   photsys_file  - Gaia EDR3 passbands (ESA/Gaia; Riello et al. 2021,
    #                   A&A 649, A3), YBC bolometric corrections (Chen et al.
    #                   2019, A&A 632, A105)
    #   imf_file      - Kroupa (2001, MNRAS 322, 231; 2002, Science 295, 82)
    #   extinction_*  - Cardelli et al. (1989) + O'Donnell (1994), R_V = 3.1;
    #                   not applied here because A_V = 0
    #   kind_LPV      - long-period-variable periods (Trabucchi et al. 2017,
    #                   ApJ 847, 139); does not affect the isochrone
    form: List[Tuple[str, str]] = [
        ("cmd_version",     "3.8"),
        ("kind_interp",     "1"),
        ("kind_postagb",    "-1"),
        ("kind_mag",        "2"),
        ("kind_dust",       "0"),
        ("output_evstage",  "1"),
        ("track_parsec",    "parsec_CAF09_v1.2S"),
        ("track_colibri",   "parsec_CAF09_v1.2S_S_LMC_08_web"),
        ("track_postagb",   "no"),
        ("n_inTPC",         "10"),
        ("eta_reimers",     "0.2"),
        ("photsys_file",    "YBC_tab_mag_odfnew/tab_mag_gaiaEDR3.dat"),
        ("photsys_version", "YBCnewVega"),
        ("dust_sourceM",    "nodustM"),
        ("dust_sourceC",    "nodustC"),
        ("extinction_av",   "0.0"),
        ("extinction_coeff", "constant"),
        ("extinction_curve", "cardelli"),
        ("kind_LPV",        "1"),
        ("imf_file",        "tab_imf/imf_kroupa_orig.dat"),
        ("isoc_isagelog",   "1"),
        ("isoc_lagelow",    f"{log_age:.2f}"),
        ("isoc_lageupp",    f"{log_age:.2f}"),
        ("isoc_dlage",      "0.0"),
        ("isoc_ismetlog",   "0"),
        ("isoc_zlow",       f"{Z:.6f}"),
        ("isoc_zupp",       f"{Z:.6f}"),
        ("isoc_dz",         "0.0"),
        ("output_kind",     "0"),
        ("submit_form",     "Submit"),
    ]
    for cf in _CGIFIELDS:
        form.append((".cgifields", cf))

    verify = True
    try:
        try:
            html = requests.post(PARSEC_URL, data=form, timeout=60, verify=verify)
        except requests.exceptions.SSLError:
            # The server does not send the intermediate CA certificate in the
            # handshake; extend the CA bundle and retry.
            verify = _extended_ca_bundle()
            html = requests.post(PARSEC_URL, data=form, timeout=60, verify=verify)
        html.raise_for_status()
        m = re.search(r'href=([^\s>]+\.dat)', html.text)
        if not m:
            return None
        dat_path = m.group(1).strip('"\'')
        dat_url = (dat_path if dat_path.startswith("http")
                   else PARSEC_BASE + "/" + dat_path.lstrip("./"))
        dat = requests.get(dat_url, timeout=30, verify=verify)
        dat.raise_for_status()
    except Exception as e:
        print(f"  Error communicating with the PARSEC service: {e}", file=sys.stderr)
        return None
    return parse_dat(dat.text)


def parse_dat(text: str) -> Optional[Tuple[np.ndarray, np.ndarray]]:
    """Parse the plain-text .dat file produced by the CMD 3.8 service."""
    header: Optional[List[str]] = None
    rows: List[List[float]] = []

    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            continue
        if line.startswith("#"):
            cand = line.lstrip("# ").split()
            if "Zini" in cand and any("mag" in c.lower() for c in cand):
                header = cand
        else:
            if header is None:
                continue
            try:
                vals = list(map(float, line.split()))
                if len(vals) == len(header):
                    rows.append(vals)
            except ValueError:
                pass

    if not header or not rows:
        return None

    arr = np.array(rows)
    col = {n: i for i, n in enumerate(header)}
    for k in ("Gmag", "G_BPmag", "G_RPmag"):
        if k not in col:
            print(f"  Error: column '{k}' not found in the PARSEC output.",
                  file=sys.stderr)
            return None

    G = arr[:, col["Gmag"]]
    BP = arr[:, col["G_BPmag"]]
    RP = arr[:, col["G_RPmag"]]
    color = BP - RP

    if "label" in col:
        # 0..5 = PMS, MS, SGB, RGB, CHeB (4, 5) - see the description above
        lab = arr[:, col["label"]].astype(int)
        keep = (lab >= 0) & (lab <= 5)
    else:
        keep = np.ones_like(G, dtype=bool)
    # G is an absolute magnitude here; the limit 18 only guards against
    # invalid rows (a choice for this exercise, not a literature value)
    valid = keep & np.isfinite(color) & np.isfinite(G) & (G < 18)
    if valid.sum() < 5:
        return None
    return color[valid], G[valid]


def write_csv(path, color, M_G, log_age, feh):
    age_myr = 10**log_age / 1e6
    with open(path, "w", encoding="utf-8", newline="") as f:
        f.write("# Source: PARSEC v1.2S, CMD 3.8 (Bressan et al. 2012; Marigo et al. 2017)\n")
        f.write(f"# log(age) = {log_age:.2f}  ({age_myr:.1f} Myr)\n")
        f.write(f"# [Fe/H] = {feh:.2f}\n")
        f.write("# Photometric system: Gaia EDR3 (G, G_BP, G_RP), "
                "YBC (Chen et al. 2019)\n")
        f.write("# Extinction A_V = 0 (isochrone in intrinsic space)\n")
        f.write(f"# Number of points: {color.size}\n")
        f.write("#\n")
        w = csv.writer(f)
        w.writerow(["BP_RP", "M_G"])
        for c, m in zip(color, M_G):
            w.writerow([f"{c:.4f}", f"{m:.4f}"])


def main(log_age, feh=0.0, output=None):
    output = output if output else f"isochrone_log{log_age:.2f}.csv"

    print(f"Downloading PARSEC isochrone "
          f"(log(age)={log_age:.2f}, [Fe/H]={feh:.2f}) ...")
    iso = fetch_isochrone(log_age, feh)
    if iso is None:
        print("Failed: the isochrone could not be downloaded.", file=sys.stderr)
        return
    color, M_G = iso
    write_csv(output, color, M_G, log_age, feh)
    print(f"Saved: {output}  ({color.size} points)")

    try:
        from google.colab import files
        files.download(output)
    except ImportError:
        pass  # not running in Colab; the file stays in the working directory

In [ ]:
main(LOG_AGE, FEH, OUTPUT or None)